# RJMCMC Synthetic Recovery (No Gibbs Refresh)

Walkthrough of the RJMCMC kernel (`inference.rjmcmc.kernel`) recovering a `FloorPlanGraph` from a synthetic point cloud: segment birth/death, junction/segment perturbation, and junction split/merge, per `docs/inference/rjmcmc_algorithm.md` sections 2-4.

**This notebook deliberately omits the Gibbs junction-position refresh** from section 5 of that document -- it is not yet implemented (see section 7's open items). Geometric refinement here relies entirely on the `junction_perturb` random-walk move, which section 5 derives is expected to mix far more slowly than a Gibbs draw from the true (Gaussian Markov Random Field) conditional. The second experiment below isolates and quantifies exactly that cost.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from inference.config import PriorConfig
from inference.evaluate import evaluate, junction_localisation_error
from inference.likelihood import AnalyticalLikelihood, AnalyticalLikelihoodParams
from inference.result import InferenceResult
from inference.rjmcmc.kernel import MoveType, RJMCMCConfig, log_posterior, step
from representation.floor_plan import FloorPlanGraph
from representation.visualisation import plot_floor_plan, plot_point_cloud_overlay
from simulator.engine import simulate
from simulator.noise_model import NoiseModel

## Ground truth floor plan and synthetic observation

A single rectangular room -- kept deliberately simple so that both experiments below are actually reachable by the implemented move set. (Recovering a topology that needs a *new* junction inserted along an existing wall's interior -- e.g. the two-room layout from `02_simulator_demo.ipynb`, which needs a T-junction partway along the top and bottom edges -- is **not** reachable by RJMCMC as currently implemented: segment birth/death only connect *existing* junctions, and junction split only duplicates an *existing* junction nearby. A `segment split` move that inserts a junction at an interior point isn't implemented yet. Keeping the ground truth to a single room sidesteps that gap so this notebook stays focused on the Gibbs question.)

In [ ]:
def build_rectangular_room(corners) -> FloorPlanGraph:
    graph = FloorPlanGraph()
    junctions = [graph.add_junction(c) for c in corners]
    for start, end in zip(junctions, junctions[1:] + junctions[:1]):
        graph.add_segment(start.id, end.id)
    return graph


TRUE_CORNERS = [(0.0, 0.0), (3.0, 0.0), (3.0, 2.0), (0.0, 2.0)]
floor_plan_true = build_rectangular_room(TRUE_CORNERS)

ax = plot_floor_plan(floor_plan_true)
ax.set_title(
    f"Ground truth: {len(floor_plan_true.junctions)} junctions, "
    f"{len(floor_plan_true.segments)} segments, area={3 * 2}m^2"
)
plt.show()

In [ ]:
scanner_positions = jnp.array([[1.5, 1.0]])  # one scanner at room centre
noise_model = NoiseModel(
    range_sigma=0.02, angular_sigma=0.01, outlier_rate=0.02, missing_rate=0.05, height_sigma=0.02
)

point_cloud = simulate(
    floor_plan_true, scanner_positions, [1.0], noise_model, jax.random.PRNGKey(0), num_rays=90
)
print(f"{point_cloud.points_3d.shape[0]} observed points")

ax = plot_point_cloud_overlay(floor_plan_true, np.asarray(point_cloud.points_3d)[:, :2])
ax.set_title("Synthetic observation D (ground truth + noisy scan)")
plt.show()

## Prior and likelihood configuration

`room_area_mean`/`room_area_std` are tuned to this room's 6 m^2 footprint; everything else is `PriorConfig`'s default. `AnalyticalLikelihoodParams` matches the noise scale used to simulate `D` above.

In [ ]:
prior_config = PriorConfig(room_area_mean=6.0, room_area_std=3.0)
likelihood = AnalyticalLikelihood(
    AnalyticalLikelihoodParams(wall_distance_sigma=0.05, inlier_weight=0.9)
)

print(
    "log posterior at ground truth:",
    f"{log_posterior(floor_plan_true, point_cloud, likelihood, prior_config):.2f}",
)

## RJMCMC driver

`inference.rjmcmc.kernel.step` performs one propose/accept-reject iteration. There is no orchestrating `run_inference()` yet (`docs/inference.md`'s `run_inference` is a planned top-level entry point across all five inference methods, not part of the RJMCMC kernel itself), so this notebook drives the loop directly and assembles the result into the shared `InferenceResult` schema.

In [ ]:
def run_rjmcmc(
    initial_graph,
    point_cloud,
    likelihood,
    prior_config,
    rjmcmc_config,
    n_iterations,
    rng_key,
    *,
    sample_every: int = 20,
    track_fn=None,
    track_every: int = 10,
):
    """Runs `step` for `n_iterations`, tracking diagnostics.

    `track_fn`, if given, is called on the current graph every `track_every`
    iterations (e.g. to record junction localisation error against a known truth)
    and its outputs collected as `(iteration, value)` pairs.
    """
    graph = initial_graph
    log_posterior_trace = []
    n_segments_trace = []
    proposed_counts = {move: 0 for move in MoveType}
    accepted_counts = {move: 0 for move in MoveType}
    samples = []
    tracked = []
    n_likelihood_evaluations = 0

    map_estimate = graph
    map_log_posterior = log_posterior(graph, point_cloud, likelihood, prior_config)

    start = time.time()
    for i in tqdm(range(n_iterations)):
        rng_key, step_key = jax.random.split(rng_key)
        result = step(graph, point_cloud, likelihood, prior_config, rjmcmc_config, step_key)
        graph = result.graph
        proposed_counts[result.move_type] += 1
        if result.accepted:
            accepted_counts[result.move_type] += 1
        if result.log_acceptance_ratio > float("-inf"):
            n_likelihood_evaluations += 2  # current + proposed

        current_lp = log_posterior(graph, point_cloud, likelihood, prior_config)
        log_posterior_trace.append(current_lp)
        n_segments_trace.append(len(graph.segments))
        if current_lp > map_log_posterior:
            map_log_posterior = current_lp
            map_estimate = graph

        if i % sample_every == 0:
            samples.append(graph)
        if track_fn is not None and i % track_every == 0:
            tracked.append((i, track_fn(graph)))
    runtime_seconds = time.time() - start

    acceptance_rates = {
        move.value: (
            accepted_counts[move] / proposed_counts[move] if proposed_counts[move] else 0.0
        )
        for move in MoveType
    }

    return graph, {
        "log_posterior_trace": jnp.array(log_posterior_trace),
        "n_segments_trace": n_segments_trace,
        "acceptance_rates": acceptance_rates,
        "samples": samples,
        "map_estimate": map_estimate,
        "map_log_posterior": map_log_posterior,
        "n_likelihood_evaluations": n_likelihood_evaluations,
        "runtime_seconds": runtime_seconds,
        "tracked": tracked,
    }


def _junction_position_std(samples, map_estimate):
    """Rough per-junction std across stored samples sharing the MAP's junction
    count -- approximate, since junction identity isn't tracked across a
    transdimensional chain; good enough as a diagnostic, not a rigorous estimate.
    """
    target_nj = len(map_estimate.junctions)
    matching = [s for s in samples if len(s.junctions) == target_nj]
    if len(matching) < 2:
        return jnp.zeros((target_nj, 2))
    arrays = jnp.stack([s.to_junction_array()[:target_nj] for s in matching])
    return jnp.std(arrays, axis=0)


def to_inference_result(final_graph, diagnostics) -> InferenceResult:
    samples = diagnostics["samples"]
    map_estimate = diagnostics["map_estimate"]
    segment_counts = [len(s.segments) for s in samples] if samples else [len(final_graph.segments)]
    return InferenceResult(
        samples=samples,
        log_weights=jnp.zeros(len(samples)),
        map_estimate=map_estimate,
        junction_position_std=_junction_position_std(samples, map_estimate),
        n_segments_distribution=jnp.array(np.bincount(segment_counts)),
        method="rjmcmc",
        n_likelihood_evaluations=diagnostics["n_likelihood_evaluations"],
        runtime_seconds=diagnostics["runtime_seconds"],
        acceptance_rates=diagnostics["acceptance_rates"],
        ess=float(len(samples)),  # naive upper bound -- no autocorrelation-based ESS yet
        log_posterior_trace=diagnostics["log_posterior_trace"],
    )

## Experiment 1: topology + geometry recovery

The chain starts from the true junctions (perturbed) with **one wall missing** -- recovering the ground truth needs one segment-birth move (topology) plus many junction-perturbation moves (geometry).

In [ ]:
def perturb_positions(graph: FloorPlanGraph, sigma: float, rng_key) -> FloorPlanGraph:
    perturbed = graph.copy()
    for junction_id in list(graph.junctions):
        rng_key, subkey = jax.random.split(rng_key)
        noise = jax.random.normal(subkey, (2,)) * sigma
        perturbed.move_junction(junction_id, graph.junctions[junction_id].position + noise)
    return perturbed


initial_graph = perturb_positions(floor_plan_true, sigma=0.15, rng_key=jax.random.PRNGKey(1))
initial_graph.remove_segment(1)  # drop the right-hand wall -- (3,0) to (3,2)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_floor_plan(floor_plan_true, ax=axes[0])
axes[0].set_title("Ground truth")
plot_floor_plan(initial_graph, ax=axes[1])
axes[1].set_title("Initial chain state (1 wall missing, positions perturbed)")
fig.tight_layout()
plt.show()

In [ ]:
rjmcmc_config = RJMCMCConfig()
N_ITERATIONS_1 = 3000

final_graph_1, diagnostics_1 = run_rjmcmc(
    initial_graph,
    point_cloud,
    likelihood,
    prior_config,
    rjmcmc_config,
    N_ITERATIONS_1,
    jax.random.PRNGKey(2),
)
result_1 = to_inference_result(final_graph_1, diagnostics_1)
print(f"runtime: {diagnostics_1['runtime_seconds']:.1f}s")
print(
    f"MAP: {len(result_1.map_estimate.junctions)} junctions, "
    f"{len(result_1.map_estimate.segments)} segments"
)

In [ ]:
result_1.plot_diagnostics()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_floor_plan(floor_plan_true, ax=axes[0])
axes[0].set_title("Ground truth")
plot_floor_plan(result_1.map_estimate, ax=axes[1])
axes[1].set_title(f"MAP estimate ({N_ITERATIONS_1} iterations, no Gibbs)")
fig.tight_layout()
plt.show()

In [ ]:
report_1 = evaluate(result_1, floor_plan_true)
print(f"segment Chamfer distance:    {report_1.chamfer_distance_segments:.3f} m")
print(f"junction localisation error: {report_1.junction_localisation_error:.3f} m")
print(f"room count accuracy:         {report_1.room_count_accuracy:.2f}")
print(f"topology accuracy:           {report_1.topology_accuracy:.2f}")

## Experiment 2: isolating the cost of no Gibbs refresh

Same room, but the chain starts from the *correct topology* -- only junction positions are wrong. This isolates exactly what section 5 of `docs/inference/rjmcmc_algorithm.md` argues: without a Gibbs draw from the true (Gaussian Markov Random Field) conditional, `junction_perturb`'s random walk is the only mechanism refining geometry, and mixes slowly. Tracking `junction_localisation_error` against the known truth every few iterations makes that convergence rate directly visible.

In [ ]:
initial_graph_2 = perturb_positions(floor_plan_true, sigma=0.2, rng_key=jax.random.PRNGKey(3))
print(
    "initial junction localisation error:",
    f"{junction_localisation_error(initial_graph_2, floor_plan_true):.3f} m",
)

N_ITERATIONS_2 = 3000
final_graph_2, diagnostics_2 = run_rjmcmc(
    initial_graph_2,
    point_cloud,
    likelihood,
    prior_config,
    rjmcmc_config,
    N_ITERATIONS_2,
    jax.random.PRNGKey(4),
    track_fn=lambda g: junction_localisation_error(g, floor_plan_true),
    track_every=10,
)
print(f"runtime: {diagnostics_2['runtime_seconds']:.1f}s")

In [ ]:
iterations, errors = zip(*diagnostics_2["tracked"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(iterations, errors)
ax.set_xlabel("iteration")
ax.set_ylabel("junction localisation error (m)")
ax.set_title("Geometric convergence via junction_perturb alone (no Gibbs refresh)")
ax.axhline(0.0, color="grey", linestyle="--", linewidth=1)
plt.show()

print(f"final junction localisation error: {errors[-1]:.3f} m")
print(f"final n_segments: {len(final_graph_2.segments)} (true: {len(floor_plan_true.segments)})")

## Summary

This exercised the full RJMCMC kernel -- all six moves from `docs/inference/rjmcmc_algorithm.md` sections 2-4 -- on synthetic recovery, and isolated the specific cost of running without the Gibbs junction-position refresh from section 5: geometric convergence depends entirely on `junction_perturb`'s random walk, which the notebook's own error trace shows is slow relative to the iteration budget.

Two follow-ups this motivates directly:

1. **Implement the Gibbs refresh** (section 5) and re-run this same notebook's Experiment 2 for a direct before/after comparison of convergence rate.
2. **A `segment split` move** that inserts a new junction along an existing wall's interior, needed to make T-junction topologies (like the two-room layout in `02_simulator_demo.ipynb`) reachable at all -- currently out of scope for every move in `inference.rjmcmc.proposals`.